In [ ]:
pip install yfinance transformers feedparser beautifulsoup4 pandas requests transformers openBB

In [ ]:
pip install pandas_ta yfinance transformers beautifulsoup4 feedparser

In [ ]:
import pandas as pd
import yfinance as yf
import datetime
import requests
import feedparser
from bs4 import BeautifulSoup
import logging
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
from openbb import obb
import pandas_ta as ta

Extensions to add: benzinga@1.4.2, bls@1.1.3, cftc@1.1.2, commodity@1.3.2, crypto@1.4.2, currency@1.4.2, derivatives@1.4.2, econdb@1.3.2, economy@1.4.3, equity@1.4.2, etf@1.4.2, federal_reserve@1.4.4, fixedincome@1.4.4, fmp@1.4.3, fred@1.4.5, imf@1.1.2, index@1.4.2, intrinio@1.4.2, news@1.4.2, oecd@1.4.2, polygon@1.4.2, regulators@1.4.3, sec@1.4.5, tiingo@1.4.2, tradingeconomics@1.4.2, us_eia@1.1.2, yfinance@1.4.7

Building...


RuntimeError: Tried to instantiate class 'list.model_fields', but it does not exist! Ensure that it is registered via torch::class_

Setup Logging

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

Phase 2: Get Fundamentals using OpenBB

In [ ]:
def get_openbb_fundamentals(ticker):
  """
  Fetch key fundamental metrics using OpenBB.
  Returns a dictionary with PE, EPS, ROE, Debt/Equity.

  """

  try:
    fundamentals = obb.equity.fundamentals(ticker)
    data = fundamentals.to_dict()
    metrics = {
        'Ticker': ticker,
        'PE': data.get('pe_ratio'),
        'EPS': data.get('eps'),
        'ROE': data.get('roe'), # Added a comma here
        'DebtToEquity': data.get('debt_to_equity')
    }
    logging.info(f"Fetched OpenBB fundmaentals for {ticker}: {metrics}")
    return metrics
  except Exception as e:
    logging.error(f"Error fetching OpenBB fundamentals for {ticker}: {e}")
    return {
        'Ticker': ticker, # Added a comma here
        'PE': None,
        "EPS": None,
        "ROE": None,
        "DebtToEquity": None
    }

Phase 1: Get stock price data

In [ ]:
def get_stock_data(ticker, period='1y', interval='1d'):
  try:
      stock = yf.Ticker(ticker)
      df = stock.history(period=period, interval=interval)
      df.reset_index(inplace=True)
      df['Ticker'] = ticker
      logging.info(f"Downloaded {len(df)} rows for {ticker}")
      return df[['Date', 'Ticker', 'Close', 'Volume']]
  except Exception as e:
      logging.error(f'Error fetching data for {ticker}: {e}')
      return pd.DataFrame()


Saving Dataframe to CSV

In [ ]:
def save_to_csv(df, filename):
  try:
    df.to_csv(filename, index=False)
    logging.info(f"Saved data to {filename}")
  except Exception as e:
    logging.error(f"Error saving {filename}: {e}")

Combine with Sentiment Data

In [ ]:
def combine_sentiment_with_price(sentiment_df, price_df):
  """

  Merge sentiment DataFrame with price DataFrame on Date and Ticker.
  Assumes sentiment_df has 'Date' and 'Ticker' columns.
  """
  # Convert 'Date' column in sentiment_df to datetime objects
  sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date'])

  # Ensure timezone is consistent for merging
  # Assuming price_df 'Date' column is timezone-aware, make sentiment_df 'Date' timezone-aware
  # You might need to adjust the timezone based on your specific data source for price_df
  if not price_df['Date'].dt.tz:
      logging.warning("Price dataframe Date column is not timezone-aware. Assuming UTC for conversion.")
      sentiment_df['Date'] = sentiment_df['Date'].dt.tz_localize('UTC')
  else:
      sentiment_df['Date'] = sentiment_df['Date'].dt.tz_localize(price_df['Date'].dt.tz)

  combined_df = pd.merge(sentiment_df, price_df, on=['Date', 'Ticker'], how='inner')
  logging.info(f"Combined dataset has {len(combined_df)} rows")
  return combined_df

Phase 3: Technical Indicators

In [ ]:
def add_technical indicators(df):
  df['RSI'] = ta.rsi(df['Close'], length=14)
  macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
  df["MACD"] = macd["MACD_12_26_9"]
  df["MACD_signal"] = macd['MACDs_12_26_9']
  df["SMA_20"] = ta.sma(df["Close"], length=20)
  df['EMA_20'] = ta.ema(df['Close'], length=20)
  logging.info("Added technical indicators")
  return df

Phase 4: Labeling logic

In [ ]:
def add_labels(df, lookahead_days=5, buy_thresh=0.02, sell_thresh=-0.02):
  df = df.sort_values('Date').reset_index(drop=True)
  df['Future_Close'] = df['Close'].Shift(-lookahead_days)
  df['Future_Return'] = (df['Future_Close'] - df['Close']) / df['Close']

  def label_row(row):
    if row['Future_Return'] >= buy_thresh:
      return 'Buy'
    elif row['Future_Return'] <= sell_thresh:
      return 'Sell'
    else:
      return 'Hold'

  df['Label'] = df.apply(label_row, axis=1)
  logging.info("Added lables based on future returns")
  return df


Finbert setup

In [ ]:
#Loads a financial domain version for BERT (FinBERT) for classifying text as positive, negative, or neutral.
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Device set to use cuda:0


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Tickers

In [ ]:
search_terms = {
    "UTHR": "United Therapeutics Corp"
}

Finviz Scraper

In [ ]:
# Pulling news headlines from Finviz
#Scrapes the latest headlines from the stock's Finviz profile page
def get_finviz_headlines(ticker):
    url = f"https://finviz.com/quote.ashx?t={ticker}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        news_table = soup.find("table", class_="fullview-news-outer")
        rows = news_table.find_all("tr") if news_table else []
        headlines = [row.a.get_text(strip=True) for row in rows if row.a]
        print(f"Pulled {len(headlines)} Finviz headlines for {ticker}")
        return headlines
    except Exception as e:
        print(f"Finviz fetch failed for {ticker}: {e}")
        return []

Google News RSS

In [ ]:
#Getting headlines from Google News RSS
#Queries Google News RSS feed using the format [TICKER] stock

def get_google_news_rss(ticker):
    query = f"{ticker} stock"
    url = f"https://news.google.com/rss/search?q={query.replace(' ', '+')}"
    try:
        feed = feedparser.parse(url)
        entries = [entry.title for entry in feed.entries]
        print(f"Pulled {len(entries)} Google RSS headlines for {ticker}")
        return entries
    except Exception as e:
        print(f"Google RSS fetch failed for {ticker}: {e}")
        return []




SEC

Getting CIK Map

In [ ]:
#Pulls public JSON file from the SEC and returns a dictionaty mapping tickers to a 10 digit CIK codes

HEADERS = {'User-Agent': 'CompanyName Contact@Company.com'} # Define HEADERS here

def get_cik_map():
    url = "https://www.sec.gov/files/company_tickers.json"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        cik_map = {
            entry["ticker"]: str(entry["cik_str"]).zfill(10)
            for entry in data.values()
        }
        return cik_map
    except Exception as e:
        print(f"Failed to fetch CIK map: {e}")
        return {}
CIK_MAP = get_cik_map()

Get SEC filings

In [ ]:
#Gets last SEC 10-Q Filings
#Pulls and parse the four most recent quarterly filings from EDGAR

def get_recent_sec_filing_texts(ticker, form_type="10-Q", count=2):
    cik = CIK_MAP.get(ticker.upper())
    if not cik:
        print(f"CIK not found for {ticker}")
        return []

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    print(f"[DEBUG] URL: {url}")
    try:
        res = requests.get(url, headers=HEADERS)
        print(f"[DEBUG] Status Code: {res.status_code}")
        res.raise_for_status()
    except Exception as e:
        print(f"Failed to get filings for {ticker}: {e}")
        return []

    data = res.json()
    filings = data.get("filings", {}).get("recent", {})
    forms = filings.get("form", [])
    documents = filings.get("primaryDocument", [])
    accession_numbers = filings.get("accessionNumber", [])

    indices = [
        i for i, (f, d) in enumerate(zip(forms, documents))
        if "10-q" in f.lower() or "10-q" in d.lower()
    ]

    print(f"[DEBUG] Matching SEC indices: {indices}")
    texts = []

    for i in indices[:count]:
        acc_num = accession_numbers[i].replace("-", "")
        doc_name = docume2

8K

In [ ]:
def get_recent_sec_filing_texts(ticker, form_type="8-K", count=2):
    cik = CIK_MAP.get(ticker.upper())
    if not cik:
        print(f"CIK not found for {ticker}")
        return []

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    print(f"[DEBUG] URL: {url}")
    try:
        res = requests.get(url, headers=HEADERS)
        print(f"[DEBUG] Status Code: {res.status_code}")
        res.raise_for_status()
    except Exception as e:
        print(f"Failed to get filings for {ticker}: {e}")
        return []

    data = res.json()
    filings = data.get("filings", {}).get("recent", {})
    forms = filings.get("form", [])
    documents = filings.get("primaryDocument", [])
    accession_numbers = filings.get("accessionNumber", [])

    indices = [
        i for i, f in enumerate(forms)
        if form_type.lower() in f.lower()
    ]

    print(f"[DEBUG] Matching SEC indices: {indices}")
    texts = []

    for i in indices[:count]:
        acc_num = accession_numbers[i].replace("-", "")
        doc_name = documents[i]
        link = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{doc_name}"
        print(f"[DEBUG] Trying SEC link: {link}")

        try:
            response = requests.get(link, headers=HEADERS)
            soup = BeautifulSoup(response.text, "html.parser")
            text = soup.get_text(separator=" ", strip=True)

            if len(text.strip()) < 100:
                continue

            texts.append(text[:1500])
        except Exception as e:
            print(f"Could not parse SEC filing for {ticker}: {e}")

    print(f"{ticker}: Parsed {len(texts)} {form_type} filings")
    return texts

Finbert Sentiment Analysis

In [ ]:
#Classifies each headline or snippet as poisitive, negative, or neutral
#Converts the result into numneric score for aggregation

def analyze_headlines_with_finbert(headlines):
    sentiments = []
    for headline in headlines:
        try:

            tokens = tokenizer.tokenize(headline)
            if len(tokens) > 512:
                tokens = tokens[:512]
                headline = tokenizer.convert_tokens_to_string(tokens)

            result = finbert(headline)[0]
            label = result["label"].lower()
            score = result["score"]
            print(f"{headline[:60]}... → {label} ({score:.4f})")

            if label == "positive":
                sentiments.append(score)
            elif label == "negative":
                sentiments.append(-score)
            else:
                sentiments.append(0.01)
        except Exception as e:
            print(f"Sentiment error on headline: {headline[:50]} — {e}")
    return sentiments

Main Execution

In [ ]:
def run_pipeline():
    today = datetime.date.today().strftime('%Y-%m-%d') # Define 'today' here
    summary_data = []
    for ticker, query in search_terms.items():
        print(f"Processing {ticker}")
        finviz_headlines = get_finviz_headlines(ticker)
        google_headlines = get_google_news_rss(ticker)
        # sec_texts = get_recent_sec_filing_texts(ticker) # Commented out SEC filings for now due to commented out get_recent_sec_filing_texts functions
        combined_texts = finviz_headlines + google_headlines # + sec_texts
        if combined_texts:
            sentiment_scores = analyze_headlines_with_finbert(combined_texts)
            avg_score = sum(sentiment_scores) / len(sentiment_scores)
            label = (
                "Buy" if avg_score > 0.1 else
                "Hold" if avg_score >= -0.1 else
                "Sell" if avg_score > -0.4 else
                "Strong Sell"
            )
            summary_data.append({
                "Date": today, # Use defined 'today'
                "Ticker": ticker,
                "Average Score": avg_score,
                "Sentiment Label": label,
            })
        else:
            print(f"No headlines found for {ticker}")
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv("sentiment_results.csv", index=False)
    print("Saved as 'sentiment_results.csv'")
    print(summary_df)
    return summary_df # Return summary_df

if __name__ == "__main__":
    sentiment_df = run_pipeline()

    for ticker in sentiment_df['Ticker']:
      price_df = get_stock_data(ticker)
      if price_df.empty:
        continue

      price_df = add_terchnical_indicators(price_df)

      sentiment_ticket_df = sentiment_df[sentiment_df['Ticker'] == ticker]
      combined_df = combine_sentiment_with_price(sentiment_ticket_df, price_df)

      combined_df = add_labels(combined_df)

      save_to_csv(combined_df, f"{ticker}_sentiment_price.csv")

Processing UTHR
Pulled 100 Finviz headlines for UTHR
Pulled 100 Google RSS headlines for UTHR
United Therapeutics Corporation to Report Second Quarter 202... → neutral (1.0000)
Are You a Momentum Investor? This 1 Stock Could Be the Perfe... → positive (0.7355)
Liquidia Receives $50M from HCRx, Accelerates YUTREPIA's Com... → positive (1.0000)
Why United Therapeutics (UTHR) is a Top Growth Stock for the... → positive (1.0000)
UBS Trims Price Target but Stays Bullish on United Therapeut... → positive (0.9986)
1 Value Stock with Impressive Fundamentals and 2 to Turn Dow... → positive (1.0000)
How to Boost Your Portfolio with Top Medical Stocks Set to B... → positive (1.0000)
Will United Therapeutics (UTHR) Beat Estimates Again in Its ... → positive (1.0000)
MRK Pins Hopes on New PAH Drug Winrevair Amid Looming Keytru... → neutral (0.9299)
Why Investors Need to Take Advantage of These 2 Medical Stoc... → neutral (0.9260)
United Therapeutics Announces World's First Bioengineered Ex... → neu

My research project, which earned you first place at the CSU student research competition, successfully investigated the correlation between stock returns and sentiment by developing a comprehensive data analysis pipeline. This pipeline effectively gathered historical stock price data using yfinance, collected news headlines from diverse sources like Finviz, Google News RSS, and SEC filings, and then employed the FinBERT model to perform sentiment analysis on these headlines, converting them into quantifiable scores. By integrating this sentiment data with technical indicators calculated using pandas_ta and applying a labeling logic based on future price movements, you were able to identify potential trading signals and demonstrate a notable short-term correlation between stock returns and the analyzed sentiment, providing a data-driven answer to your research question.